In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
# %matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import GridSearchCV

In [2]:
data = pd.read_csv(r'C:\Users\Lenovo\Desktop\DEPI(R_2)\ML_git\GIZ4_AIS2_S1_Ml\src\ML\Session_7\Code\creditcard.csv')
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
data = data.drop(['Time'], axis=1)

In [4]:
x = data.drop(['Class'], axis=1)
y = data['Class']

In [5]:
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.2, random_state=42)

In [6]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [7]:
gbm_param = {
    'n_estimators': [10],
    'learning_rate': [0.01, 0.1, 0.05],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.7, 0.8, 0.9],
    'min_samples_split': [2, 5, 10]
}

Verbose -----> log of data if i need to print it set verbose = 1

In [8]:
gb_grid = GridSearchCV(estimator=GradientBoostingClassifier(), 
                        param_grid=gbm_param,
                        scoring='roc_auc',
                        cv=3, 
                        n_jobs=-1, 
                        verbose=2)


gb_cv = gb_grid.fit(x_train, y_train)

Fitting 3 folds for each of 108 candidates, totalling 324 fits


In [9]:
print ("The best paramter combination is ")
print(gb_cv.best_params_)  #gets best estimator

# Prediction Using the Model
y_pred_gbm = gb_cv.best_estimator_.predict(x_test)
cm_gbm = confusion_matrix(y_test, y_pred_gbm)
print(cm_gbm)
print(classification_report(y_test, y_pred_gbm, target_names=["Safe", "Fraud"]))

# Calculate sensitivity, specificity, and accuracy
total_gbm = sum(sum(cm_gbm))
accuracy_gbm = (cm_gbm[0,0] + cm_gbm[1,1]) / total_gbm
print('Accuracy (GBM): ', accuracy_gbm)

sensitivity_gbm = cm_gbm[0,0] / (cm_gbm[0,0] + cm_gbm[0,1])
print('Sensitivity (GBM): ', sensitivity_gbm)

specificity_gbm = cm_gbm[1,1] / (cm_gbm[1,0] + cm_gbm[1,1])
print('Specificity (GBM): ', specificity_gbm)


The best paramter combination is 
{'learning_rate': 0.01, 'max_depth': 6, 'min_samples_split': 10, 'n_estimators': 10, 'subsample': 0.7}
[[227459      0]
 [   381      6]]
              precision    recall  f1-score   support

        Safe       1.00      1.00      1.00    227459
       Fraud       1.00      0.02      0.03       387

    accuracy                           1.00    227846
   macro avg       1.00      0.51      0.51    227846
weighted avg       1.00      1.00      1.00    227846

Accuracy (GBM):  0.9983278179120986
Sensitivity (GBM):  1.0
Specificity (GBM):  0.015503875968992248


In [10]:
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [11]:
xgbm_param = {
    'n_estimators': [10],
    'learning_rate': [0.01, 0.1, 0.05],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}


xgb_grid = GridSearchCV(estimator=XGBClassifier(), 
                        param_grid=xgbm_param,
                        scoring='roc_auc',
                        cv=3, 
                        n_jobs=-1, 
                        verbose=2)

xgb_cv = xgb_grid.fit(x_train, y_train)

Fitting 3 folds for each of 108 candidates, totalling 324 fits


In [12]:
print ("The best paramter combination is ")
print(xgb_cv.best_params_)  #gets best estimator

# Prediction Using the Model
y_pred_xgb = xgb_cv.best_estimator_.predict(x_test)
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print(cm_xgb)
print(classification_report(y_test, y_pred_xgb, target_names=["Safe", "Fraud"]))

# Calculate sensitivity, specificity, and accuracy
total_xgb = sum(sum(cm_xgb))
accuracy_xgb = (cm_xgb[0,0] + cm_xgb[1,1]) / total_xgb
print('Accuracy (XGB): ', accuracy_xgb)

sensitivity_xgb = cm_xgb[0,0] / (cm_xgb[0,0] + cm_xgb[0,1])
print('Sensitivity (XGB): ', sensitivity_xgb)

specificity_xgb = cm_xgb[1,1] / (cm_xgb[1,0] + cm_xgb[1,1])
print('Specificity (XGB): ', specificity_xgb)


The best paramter combination is 
{'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 10, 'subsample': 0.8}
[[227459      0]
 [   387      0]]
              precision    recall  f1-score   support

        Safe       1.00      1.00      1.00    227459
       Fraud       0.00      0.00      0.00       387

    accuracy                           1.00    227846
   macro avg       0.50      0.50      0.50    227846
weighted avg       1.00      1.00      1.00    227846

Accuracy (XGB):  0.9983014843359111
Sensitivity (XGB):  1.0
Specificity (XGB):  0.0


c:\Users\Lenovo\Desktop\DEPI(R_2)\ML_git\GIZ4_AIS2_S1_Ml\.env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\Desktop\DEPI(R_2)\ML_git\GIZ4_AIS2_S1_Ml\.env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\Desktop\DEPI(R_2)\ML_git\GIZ4_AIS2_S1_Ml\.env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

In [13]:
from lightgbm import LGBMClassifier

lgbm_param = {
    'n_estimators': [10],
    'learning_rate': [0.01, 0.1, 0.05],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.7, 0.8, 0.9],
}

lgb_grid = GridSearchCV(estimator=LGBMClassifier(),
                        param_grid=lgbm_param,
                        scoring='roc_auc',
                        cv=3,
                        n_jobs=-1,
                        verbose=2)

lgb_cv = lgb_grid.fit(x_train, y_train)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
[LightGBM] [Info] Number of positive: 105, number of negative: 56856
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004117 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7395
[LightGBM] [Info] Number of data points in the train set: 56961, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001843 -> initscore=-6.294317
[LightGBM] [Info] Start training from score -6.294317
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

In [14]:
print ("The best paramter combination is ")
print(lgb_cv.best_params_)  #gets best estimator

# Prediction Using the Model
y_pred_lgb = lgb_cv.best_estimator_.predict(x_test)
cm_lgb = confusion_matrix(y_test, y_pred_lgb)
print(cm_lgb)
print(classification_report(y_test, y_pred_lgb, target_names=["Safe", "Fraud"]))

# Calculate sensitivity, specificity, and accuracy
total_lgb = sum(sum(cm_lgb))
accuracy_lgb = (cm_lgb[0,0] + cm_lgb[1,1]) / total_lgb
print('Accuracy (LGB): ', accuracy_lgb)

sensitivity_lgb = cm_lgb[0,0] / (cm_lgb[0,0] + cm_lgb[0,1])
print('Sensitivity (LGB): ', sensitivity_lgb)

specificity_lgb = cm_lgb[1,1] / (cm_lgb[1,0] + cm_lgb[1,1])
print('Specificity (LGB): ', specificity_lgb)


The best paramter combination is 
{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 10, 'subsample': 0.7}
[[227459      0]
 [   387      0]]
              precision    recall  f1-score   support

        Safe       1.00      1.00      1.00    227459
       Fraud       0.00      0.00      0.00       387

    accuracy                           1.00    227846
   macro avg       0.50      0.50      0.50    227846
weighted avg       1.00      1.00      1.00    227846

Accuracy (LGB):  0.9983014843359111
Sensitivity (LGB):  1.0
Specificity (LGB):  0.0


c:\Users\Lenovo\Desktop\DEPI(R_2)\ML_git\GIZ4_AIS2_S1_Ml\.env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\Desktop\DEPI(R_2)\ML_git\GIZ4_AIS2_S1_Ml\.env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\Desktop\DEPI(R_2)\ML_git\GIZ4_AIS2_S1_Ml\.env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

In [15]:
from catboost import CatBoostClassifier

catb_param = {
    'iterations': [10],
    'learning_rate': [0.01, 0.1, 0.05],
    'depth': [3, 4, 5, 6],
    'subsample': [0.7, 0.8, 0.9],
}
catb_grid = GridSearchCV(estimator=CatBoostClassifier(),
                        param_grid=catb_param,
                        scoring='roc_auc',
                        cv=3,
                        n_jobs=-1,
                        verbose=2)

catb_cv = catb_grid.fit(x_train, y_train)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
0:	learn: 0.5223816	total: 152ms	remaining: 1.36s
1:	learn: 0.4002871	total: 163ms	remaining: 654ms
2:	learn: 0.2943732	total: 174ms	remaining: 407ms
3:	learn: 0.2253929	total: 185ms	remaining: 278ms
4:	learn: 0.1679236	total: 194ms	remaining: 194ms
5:	learn: 0.1314415	total: 203ms	remaining: 136ms
6:	learn: 0.1000784	total: 213ms	remaining: 91.4ms
7:	learn: 0.0766572	total: 223ms	remaining: 55.6ms
8:	learn: 0.0587106	total: 231ms	remaining: 25.6ms
9:	learn: 0.0457404	total: 240ms	remaining: 0us


In [16]:
print ("The best paramter combination is ")
print(catb_cv.best_params_)  #gets best estimator

# Prediction Using the Model
y_pred_catb = catb_cv.best_estimator_.predict(x_test)
cm_catb = confusion_matrix(y_test, y_pred_catb)
print(cm_catb)
print(classification_report(y_test, y_pred_catb, target_names=["Safe", "Fraud"]))

# Calculate sensitivity, specificity, and accuracy
total_catb = sum(sum(cm_catb))
accuracy_catb = (cm_catb[0,0] + cm_catb[1,1]) / total_catb
print('Accuracy (CATB): ', accuracy_catb)

sensitivity_catb = cm_catb[0,0] / (cm_catb[0,0] + cm_catb[0,1])
print('Sensitivity (CATB): ', sensitivity_catb)

specificity_catb = cm_catb[1,1] / (cm_catb[1,0] + cm_catb[1,1])
print('Specificity (CATB): ', specificity_catb)


The best paramter combination is 
{'depth': 5, 'iterations': 10, 'learning_rate': 0.05, 'subsample': 0.7}
[[227459      0]
 [   372     15]]
              precision    recall  f1-score   support

        Safe       1.00      1.00      1.00    227459
       Fraud       1.00      0.04      0.07       387

    accuracy                           1.00    227846
   macro avg       1.00      0.52      0.54    227846
weighted avg       1.00      1.00      1.00    227846

Accuracy (CATB):  0.9983673182763797
Sensitivity (CATB):  1.0
Specificity (CATB):  0.03875968992248062


In [17]:
from sklearn.ensemble import AdaBoostClassifier

ada_param = {
    'n_estimators': [10], 
    'learning_rate': [0.01, 0.1, 0.05]
}
ada_grid = GridSearchCV(estimator=AdaBoostClassifier(),
                        param_grid=ada_param,
                        scoring='roc_auc',
                        cv=3,
                        n_jobs=-1,
                        verbose=2)

ada_cv = ada_grid.fit(x_train, y_train)

Fitting 3 folds for each of 3 candidates, totalling 9 fits


In [18]:
print ("The best paramter combination is ")
print(ada_cv.best_params_)  #gets best estimator

# Prediction Using the Model
y_pred_ada = ada_cv.best_estimator_.predict(x_test)
cm_ada = confusion_matrix(y_test, y_pred_ada)
print(cm_ada)
print(classification_report(y_test, y_pred_ada, target_names=["Safe", "Fraud"]))

# Calculate sensitivity, specificity, and accuracy
total_ada = sum(sum(cm_ada))
accuracy_ada = (cm_ada[0,0] + cm_ada[1,1]) / total_ada
print('Accuracy (ADA): ', accuracy_ada)

sensitivity_ada = cm_ada[0,0] / (cm_ada[0,0] + cm_ada[0,1])
print('Sensitivity (ADA): ', sensitivity_ada)

specificity_ada = cm_ada[1,1] / (cm_ada[1,0] + cm_ada[1,1])
print('Specificity (ADA): ', specificity_ada)


The best paramter combination is 
{'learning_rate': 0.05, 'n_estimators': 10}
[[227427     32]
 [   168    219]]
              precision    recall  f1-score   support

        Safe       1.00      1.00      1.00    227459
       Fraud       0.87      0.57      0.69       387

    accuracy                           1.00    227846
   macro avg       0.94      0.78      0.84    227846
weighted avg       1.00      1.00      1.00    227846

Accuracy (ADA):  0.9991222141270858
Sensitivity (ADA):  0.9998593153051759
Specificity (ADA):  0.5658914728682171
